# 第 7 课 · 调试 —— 学会"抓 bug"

### 学习目标
1. 认识 4 类最高频的初级错误
2. 掌握 3 个调试手段：打印、断点、异常信息
3. 养成"先读报错，再改代码"的习惯

### 一句话理解
调试不是"猜哪里错了"，而是"用证据定位到底哪里错了"。
报错信息不是敌人，它是 Python 在帮你指路。


## 一、4 类最高频的初级错误

1) 手误（拼写错误）：变量名少个字母、函数名打错
     pritn('hello')          -> NameError: name 'pritn' is not defined

2) 多个 print 挤在一行，或者忘了换行
     print('a') print('b')   -> SyntaxError: invalid syntax
     正解：写成两行，或用分号（不推荐）print('a'); print('b')

3) 缩进错误：空格和 Tab 混用、该缩进的地方没缩进
     if True:
     print('x')              -> IndentationError: expected an indented block

4) 标点符号：用了中文的引号、括号、逗号、冒号
     print（'hello'）        -> SyntaxError: invalid character '（'
     这是中文输入法下写代码最大的坑，看到 SyntaxError 先检查标点。


In [ ]:
print('hello world1')
print('hello world2')       # 打印 hello world2
print('hello world3')
print('hello world4')


## 二、调试手段 1：打印关键变量（最快、最常用）

在"你觉得有问题的地方"前后各打一次，看值是怎么变化的。


In [ ]:
def divide(a, b):
    print(f'[调试] 收到参数 a={a!r}, b={b!r}')   # !r 用 repr 输出，能看清空串/空格/换行
    result = a / b
    print(f'[调试] 计算结果 result={result!r}')
    return result

divide(10, 3)

# 【进阶技巧】用 f-string 的 = 语法直接打印"表达式 = 值"，省掉手写标签
x = 42
y = 7
print(f'{x = }, {y = }, {x * y = }')      # x = 42, y = 7, x * y = 294
# 这个写法是 Python 3.8+ 的特性，调试时非常好用。
#
# 【专业做法】用 logging 模块代替 print，可以随时关掉、可以带时间戳和级别：
import logging
logging.basicConfig(level=logging.DEBUG, format='%(levelname)s | %(message)s')
logging.debug('这条只在调试时看得到')
logging.info('这条是正常信息')
# 上线时把 level 改成 WARNING，debug 级别的输出就自动消失了，不用一行行删 print。


## 三、调试手段 2：断点调试（VS Code 里最直观）

步骤：
  1) 在代码行号左边的灰色空白处【单击】，会出现一个红点，这就是断点（breakpoint）
  2) 按 F5 启动调试（第一次会让你选 Python 调试器）
  3) 程序会停在断点那一行【执行之前】，此时可以在左侧"变量"面板看到所有变量的值
  4) 常用按钮：
       F10  单步跳过（执行当前行，遇到函数调用不进去）
       F11  单步进入（进入被调用的函数内部）
       F5   继续运行到下一个断点
       Shift+F5  停止调试
  5) 还可以在"监视"面板里输入表达式（比如 x * y），实时看它的值

断点调试比 print 强在哪？
  不用改代码、不产生多余输出，能在任意时刻查看【所有】变量的状态，
  还能往回看调用栈——复杂逻辑排查必须靠它。


In [ ]:
def buggy_sum(numbers):
    total = 0
    for n in numbers:
        total = total + n
    return total

print('在下面这行打个断点，然后按 F5，看 total 一步步怎么变大：')
print(buggy_sum([1, 2, 3, 4, 5]))


## 四、调试手段 3：读懂异常信息（最被低估的技能）

Python 的报错信息要【从下往上】读：

  Traceback (most recent call last):
    File "demo.py", line 12, in <module>
      main()
    File "demo.py", line 8, in main
      print(1 / 0)
  ZeroDivisionError: division by zero

  最后一行：错误【类型】+ 错误【原因】   -> ZeroDivisionError: division by zero
  往上几行：错误发生的【规律路径】（调用栈），最下面那一条就是你代码里出事的位置
  看 File 后面的行号和函数名，直接跳过去改。


In [ ]:
def main_demo():
    try:
        print(1 / 0)
    except ZeroDivisionError as e:
        # 拿到异常对象后，打印它的类型和说明，方便定位
        print(f'抓到异常：{type(e).__name__} -> {e}')

main_demo()

# 【重要】不要用"裸 except"吞掉所有错误
#   try:
#       ...
#   except:          <- 不推荐：什么都抓，连你自己写错变量名也一起抓了，问题被藏起来
#       pass
# 正确姿势：能说出会抛什么异常就写具体类型；实在要兜底就 except Exception as e 并且打印出来。


## 五、五步调试法（遇到 bug 时照这个顺序走）

1. 先看报错最后一行，确定异常类型
2. 看行号，跳到出事的那一行
3. 检查这一行用到的每个变量的值（print 或断点）
4. 如果值是错的，往上游追一步：它是在哪里被算出来的
5. 重复第 4 步，直到找到"第一个错的地方"——往往真正的问题不在报错行

【心法】能复现才好修。先把问题稳定复现出来，再动手改。


## 本课修正说明

原始笔记末尾有一行 `from pymysql import Connection`，它和"调试"这个主题没关系，
是写笔记时误粘进来的。这里已删除，避免造成"导入报错但不知道为什么"的困惑。
数据库相关的内容已经挪到 mysql/mysql.ipynb 里去了。


## 上手练习

练习 1：下面这段代码有 3 个错误，逐个找出来并修好
        def add(a, b)
          return a + b
        print(add(1, 2))

练习 2：故意写一段 1/0 的代码，完整读一遍 Traceback，说出每一行的含义

练习 3：用 f-string 的 = 语法，在循环里打印每一轮 i 和累加值的变化，
        观察"累加"是怎么一步步发生的

练习 4：给一段你自己写过的、跑不通的代码，用"五步调试法"定位问题
